练习一：同步任务管理
实现以下函数：

def add_task(
    tasks: list[dict[str, object]],
    title: str,
) -> dict[str, object]:
    ...

要求：

标题不能为空
自动生成递增 ID
默认 completed=False
为函数添加完整类型注解

In [ ]:
##  练习一：同步任务管理
def add_task(
    tasks: list[dict[str, object]],
    title: str,
) -> dict[str, object]:
    """创建任务并添加到任务列表。"""
    clean_title = title.strip()
    if not clean_title:
        raise ValueError("任务标题不能为空")

    existing_ids = [
        task["id"]
        for task in tasks
        if isinstance(task.get("id"), int)
    ]
    next_id = max(existing_ids, default=0) + 1

    task = {
        "id": next_id,
        "title": clean_title,
        "completed": False,
    }
    tasks.append(task)
    return task


# 示例：创建任务，ID 会自动递增
my_tasks: list[dict[str, object]] = []
first_task = add_task(my_tasks, "  学习 Python  ")
second_task = add_task(my_tasks, "学习 FastAPI")

print(first_task)
print(second_task)
print(my_tasks)

# 基本验证
assert first_task == {"id": 1, "title": "学习 Python", "completed": False}
assert second_task["id"] == 2
assert len(my_tasks) == 2

try:
    add_task(my_tasks, "   ")
except ValueError as error:
    print(f"校验通过：{error}")
else:
    raise AssertionError("空标题应该抛出 ValueError")

练习二：异常设计
为任务管理器增加：

TaskNotFoundError
InvalidTaskTitleError
查询不存在任务时抛出明确异常
编写测试验证异常类型和错误信息

In [ ]:
class TaskNotFoundError(Exception):
    """自定义异常类，用于表示任务未找到的情况。"""
    pass

class InvalidTaskTitleError(Exception):
  """自定义异常类，用于表示任务标题无效的情况。"""
  pass

class TaskManager:
    """任务管理器类，支持添加、删除、更新和查询任务。"""
    def get_task(
      tasks: list[dict[str, object]],
      title: str,
    ) -> dict[str, object]:
        """获取任务。"""
        clean_title = title.strip()
        if not clean_title:
          raise InvalidTaskTitleError("任务标题不能为空")
        for  task in tasks:
          if task.get("title") == clean_title:
            return task
        raise TaskNotFoundError("任务未找到")
      
      
    def add_task(
        self,
        tasks: list[dict[str, object]],
        title: str,
    ) -> dict[str, object]:
        """创建任务并添加到任务列表。"""
        clean_title = title.strip()
        if not clean_title:
          raise InvalidTaskTitleError("任务标题不能为空")
        existing_ids = [
            task["id"]
            for task in tasks
            if isinstance(task.get("id"), int)
        ]
        next_id = max(existing_ids, default=0) + 1

        task = {
            "id": next_id,
            "title": clean_title,
            "completed": False,
        }
        tasks.append(task)
        return task
    

练习四：综合练习
实现一个异步“Agent 上下文收集器”：

用户问题
  -> 并发读取用户信息
  -> 并发读取历史对话
  -> 并发读取知识库摘要
  -> 合并结果
  -> 返回结构化上下文

要求：

每个外部调用都有类型注解
每个调用有超时
失败时保留错误信息，不要静默忽略
使用 dataclass 或 Pydantic 表示结果
为正常、超时和部分失败场景编写测试

In [ ]:
import asyncio
from pydantic import BaseModel


class UserInfo(BaseModel):
    name: str
    age: int
    email: str


class HistoryItem(BaseModel):
    action: str
    timestamp: str


class KnowledgeItem(BaseModel):
    topic: str
    content: str


class AgentContext(BaseModel):
    user_info: UserInfo | None = None
    history: list[HistoryItem] = []
    knowledge: dict[str, KnowledgeItem] = {}
    errors: dict[str, str] = {}


async def collect_context() -> AgentContext:
    results = await asyncio.gather(
        get_user_info(),
        get_history(),
        get_knowledge(),
        return_exceptions=True,
    )

    user_info_result, history_result, knowledge_result = results
    errors: dict[str, str] = {}

    user_info = None
    if isinstance(user_info_result, Exception):
        errors["user_info"] = str(user_info_result)
    else:
        user_info = user_info_result

    history: list[HistoryItem] = []
    if isinstance(history_result, Exception):
        errors["history"] = str(history_result)
    else:
        history = history_result

    knowledge: dict[str, KnowledgeItem] = {}
    if isinstance(knowledge_result, Exception):
        errors["knowledge"] = str(knowledge_result)
    else:
        knowledge = knowledge_result

    return AgentContext(
        user_info=user_info,
        history=history,
        knowledge=knowledge,
        errors=errors,
    )


try:
    context = await asyncio.wait_for(
        collect_context(),
        timeout=3,
    )
    print(context)
except TimeoutError:
    print("上下文收集整体超时")